In [1]:
%load_ext autoreload
%autoreload 2


In [1]:
from transformers import AutoModel, AutoConfig
from colpali_engine.models.late_interaction.colpali_architecture import ColPali
from transformers import GemmaForCausalLM,PaliGemmaForConditionalGeneration
import torch
model = "vidore/colpali-v1.2-merged"
config = AutoConfig.from_pretrained(model)
model = ColPali.from_pretrained(model, config=config, device_map="cuda").eval()


/home/akshay/miniconda3/envs/optimum/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# import torch
# torch_input = model.dummy_inputs
# onnx_program = torch.onnx.export(model,torch_input,f="colpali/colpali-v1.2-merged.onnx", input_names=['input_ids'],output_names=['logits'], dynamo=False)

In [4]:
from optimum.exporters import TasksManager
from pathlib import Path
from optimum.exporters.onnx.convert import export

onnx_path = Path("colpali/model.onnx")

onnx_config_constructor = TasksManager.get_exporter_config_constructor("onnx", model, task="feature-extraction")
onnx_config = onnx_config_constructor(model.config)
onnx_inputs, onnx_outputs = export(model, onnx_config, onnx_path, 14, device="cuda")

Passing the argument `library_name` to `get_supported_tasks_for_model_type` is required, but got library_name=None. Defaulting to `transformers`. An error will be raised in a future version of Optimum if `library_name` is not provided.
Using framework PyTorch: 2.4.1+cu121


torch.Size([2, 1040])
tensor([[257152, 257152, 257152,  ..., 119965,  49812,  21215],
        [257152, 257152, 257152,  ..., 183458,  49402,  72937]])


/home/akshay/miniconda3/envs/optimum/lib/python3.11/site-packages/transformers/models/paligemma/modeling_paligemma.py:494: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if inputs_embeds[special_image_mask].numel() != image_features.numel():
/home/akshay/miniconda3/envs/optimum/lib/python3.11/site-packages/transformers/models/paligemma/modeling_paligemma.py:377: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/home/akshay/miniconda3/envs/optimum/lib/python3.11/site-packages/transformers/models/gemma/modeling_gemma.py:857: TracerWa

InvalidGraph: [ONNXRuntimeError] : 10 : INVALID_GRAPH : Load model from colpali/model.onnx failed:This is an invalid model. Type Error: Type 'tensor(bfloat16)' of input parameter (/model/Cast_1_output_0) of operator (Conv) in node (/model/vision_tower/vision_model/embeddings/patch_embedding/Conv) is invalid.

In [2]:
from colpali_engine.utils.processing_utils.colpali_processing_utils import process_images, process_queries
from transformers import AutoProcessor
import requests
from typing import cast
from PIL import Image
processor = AutoProcessor.from_pretrained( "vidore/colpali-v1.2-merged")
image= Image.open(requests.get("https://res.cloudinary.com/dltwftrgc/image/upload/v1722427969/Blogs/yolo_clip/cover_2_kmmtfe.jpg",stream=True).raw)
inputs = process_images(processor,[image]).to("cuda")


In [7]:
inputs = process_queries(processor,["What is the image?"]).to("cuda")

In [9]:
model(**inputs)

TypeError: ColPali.forward() missing 1 required positional argument: 'pixel_values'

In [3]:
from transformers import AutoTokenizer, AutoProcessor
from PIL import Image
import requests
image= Image.open(requests.get("https://res.cloudinary.com/dltwftrgc/image/upload/v1722427969/Blogs/yolo_clip/cover_2_kmmtfe.jpg",stream=True).raw)

tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-mix-448")
processor = AutoProcessor.from_pretrained("google/paligemma-3b-mix-448")
inputs = processor(text=["What is the image?"], images=[image], return_tensors="pt").to("cuda")

model(**inputs)

TypeError: _forward_unimplemented() got an unexpected keyword argument 'input_ids'

In [1]:
from optimum.onnxruntime import ORTModelForCausalLM
model = ORTModelForCausalLM.from_pretrained("colpali")

/home/akshay/miniconda3/envs/optimum/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IndexError: list index out of range

In [10]:
model.run([model.get_outputs()[0].name],{'input_ids':inputs['input_ids'].numpy(),'attention_mask':inputs['attention_mask'].numpy()})

ValueError: Required inputs (['position_ids', 'onnx::Neg_3']) are missing from input feed (['input_ids', 'attention_mask']).

In [13]:
from optimum.pipelines import pipeline

pip = pipeline(task="text-generation", accelerator="ort", model="colpali/colpali-v1.2-merged.onnx")
pip(image)

OSError: It looks like the config file at 'colpali/colpali-v1.2-merged.onnx' is not a valid JSON file.

In [1]:
import onnxruntime as ort

sess = ort.InferenceSession("colpali/model.onnx")


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

In [4]:
input = tokenizer("Hello world", return_tensors="pt")

In [2]:
print([inp.name for inp in sess.get_inputs()])

['input_ids', 'attention_mask', 'position_ids', 'onnx::Neg_3']


In [5]:
pred=sess.run([sess.get_outputs()[0].name],{'input_ids':inputs['input_ids'].numpy(), 'attention_mask':inputs['attention_mask'].numpy()})


ValueError: Required inputs (['position_ids', 'onnx::Neg_3']) are missing from input feed (['input_ids', 'attention_mask']).

In [ ]:
sess.get_inputs()[0].name